# EDA with SQL — required rubric queries
The IBM SQL assignment historically uses a separate `SPACEXTBL` dataset containing `Mission_Outcome`, `Landing_Outcome`, `Booster_Version`, etc. Download it independently to preserve the official query semantics.

In [ ]:
import pandas as pd, sqlite3
from pathlib import Path
D=Path('../data');O=Path('../outputs');O.mkdir(exist_ok=True)
url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/Spacex.csv'
try: df=pd.read_csv(url)
except Exception:
    path=D/'Spacex.csv'
    if not path.exists(): raise FileNotFoundError('Download the IBM Spacex.csv dataset into data/Spacex.csv')
    df=pd.read_csv(path)
print(df.columns.tolist());display(df.head())
conn=sqlite3.connect(':memory:');df.to_sql('SPACEXTBL',conn,index=False,if_exists='replace')

In [ ]:
# Column names differ slightly between published copies: normalize names to SQL-friendly keys.
cols={c:''.join(ch if ch.isalnum() else '_' for ch in c.strip()).strip('_').upper() for c in df.columns}
df=df.rename(columns=cols);df.to_sql('SPACEXTBL',conn,index=False,if_exists='replace')
print('SQL columns:',df.columns.tolist())
def query(sql,label):
    print('\n'+label+'\n'+sql);result=pd.read_sql_query(sql,conn);display(result)
    result.to_csv(O/('sql_'+str(len(list(O.glob('sql_*.csv')))+1).zfill(2)+'.csv'),index=False)
    return result

In [ ]:
# Standard IBM capstone SQL queries. Review printed schema before running if column labels differ.
queries=[
('Launch site names',"SELECT DISTINCT LAUNCH_SITE FROM SPACEXTBL"),
('CCA launches',"SELECT * FROM SPACEXTBL WHERE LAUNCH_SITE LIKE 'CCA%' LIMIT 5"),
('NASA payload',"SELECT SUM(PAYLOAD_MASS__KG_) AS TOTAL_KG FROM SPACEXTBL WHERE CUSTOMER LIKE '%NASA%'"),
('Average F9 v1.1 payload',"SELECT AVG(PAYLOAD_MASS__KG_) AS AVG_KG FROM SPACEXTBL WHERE BOOSTER_VERSION LIKE 'F9 v1.1%'"),
('First ground pad landing',"SELECT MIN(DATE) AS FIRST_LANDING FROM SPACEXTBL WHERE LANDING_OUTCOME LIKE 'Success (ground pad)%'"),
('Drone ship landings, 4–6t',"SELECT DISTINCT BOOSTER_VERSION FROM SPACEXTBL WHERE LANDING_OUTCOME LIKE 'Success (drone ship)%' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000"),
('Mission outcomes',"SELECT MISSION_OUTCOME,COUNT(*) AS N FROM SPACEXTBL GROUP BY MISSION_OUTCOME"),
('Maximum payload boosters',"SELECT DISTINCT BOOSTER_VERSION FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)"),
('2015 failed drone ship landings',"SELECT DATE,BOOSTER_VERSION,LAUNCH_SITE,LANDING_OUTCOME FROM SPACEXTBL WHERE DATE LIKE '2015%' AND LANDING_OUTCOME LIKE 'Failure (drone ship)%'"),
('Landing outcome ranking',"SELECT LANDING_OUTCOME,COUNT(*) AS N FROM SPACEXTBL WHERE DATE BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY LANDING_OUTCOME ORDER BY N DESC")]
for label,sql in queries:
    try:query(sql,label)
    except Exception as e:print('Query needs column/schema adjustment:',label,e)